# Demand Characteristics Estimator

**Pipeline stage.** Consumes one `clean_master.csv` matching a fixed 12-column contract; produces
per-SKU estimates of demand volatility, chronic forecast bias (per lag) and random forecast error.

**Run all, top to bottom.** No cell needs to be run out of order, and no cell depends on state from
a previous session. Upload `clean_master.csv` to the session (left sidebar → Files) and run.

**Outputs written to disk**

| File | Purpose | Stability |
|---|---|---|
| `demand_characteristics.csv` | The estimates. Read programmatically by later stages. | **Fixed contract — 9 columns, never grows.** |
| `censoring_diagnostics.csv` | Per-SKU evidence about whether the censoring correction is trustworthy on this dataset. | Free to evolve as new checks are added. |

Diagnostics live in the second file precisely so that adding a new check never changes the
interface other code depends on.

**Design principle: the notebook prints evidence, not verdicts.** Where the evidence does not
settle a question, it says `INCONCLUSIVE` rather than forcing a call. Every threshold is derived
from the dataset being analysed; nothing is calibrated by eye from a previous run.

In [ ]:
# =========================================================================
# CELL 0 — CONFIG
# =========================================================================
# Every tunable parameter in the notebook lives here. Switching datasets
# should mean editing this cell only. Each entry states whether it is derived
# from data or is a judgement call.

# ---- input / output ------------------------------------------------------
CSV_NAME = "clean_master.csv"
OUT_PRIMARY = "demand_characteristics.csv"      # fixed downstream contract
OUT_DIAGNOSTICS = "censoring_diagnostics.csv"   # free to evolve

# ---- estimator -----------------------------------------------------------
LAGS = (1, 2, 3)                 # forecast horizons present in the schema

# JUDGEMENT. Minimum uncensored months before any number is reported for a
# SKU. Statistical convention (locate a mean and a variance with room to
# spare), not derived from data. Below this the estimate is returned as NaN
# with a reason, never as a fallback value.
MIN_UNCENSORED = 4

# JUDGEMENT, off by default. Optional softer censoring rule: if set to q in
# (0,1), a month also counts as censored when closing stock is at or below
# that SKU's own q-quantile of strictly positive closing stock. Left as None
# because the data cannot tell us where "near zero" begins — that call should
# be explicit, not buried.
NEAR_ZERO_STOCK_QUANTILE = None

# ---- diagnostics ---------------------------------------------------------
# JUDGEMENT. How many standard errors a raw (uncorrected) bias estimate must
# sit away from zero before it counts as meaningfully signed rather than
# indistinguishable from zero. 1.0 is deliberately permissive: it is a
# screening test, not a significance test.
BIAS_SE_MULTIPLIER = 1.0

# DERIVED-BY-RULE. A censored month counts as a genuinely BINDING stockout if
# its shipment ratio (shipped / available) sits at or above this quantile of
# the SAME SKU's shipment ratios in its non-censored months. The comparison
# distribution comes from the SKU itself, so no absolute cut-off is imposed.
BINDING_RATIO_QUANTILE = 0.90

# DERIVED-BY-RULE. Materiality threshold for the censoring sensitivity test:
# SKUs with at least this many censored months are adjudicated. Computed in
# Cell 4 as the median censored-month count among SKUs that have any
# stockouts at all — so it adapts to how censored the dataset actually is.
# Set to a number here only to override that rule.
CENS_MATERIALITY_OVERRIDE = None

# ---- appendix ------------------------------------------------------------
# The appendix validates the METHOD on simulated data with known answers. It
# touches none of your data. Set False to skip it.
RUN_METHOD_VALIDATION = True

print("Config loaded.")
print(f"  input                    : {CSV_NAME}")
print(f"  primary output           : {OUT_PRIMARY}")
print(f"  diagnostics output       : {OUT_DIAGNOSTICS}")
print(f"  lags                     : {LAGS}")
print(f"  min_uncensored           : {MIN_UNCENSORED}")
print(f"  near_zero_stock_quantile : {NEAR_ZERO_STOCK_QUANTILE}")
print(f"  bias_se_multiplier       : {BIAS_SE_MULTIPLIER}")
print(f"  binding_ratio_quantile   : {BINDING_RATIO_QUANTILE}")

In [ ]:
# =========================================================================
# CELL 1 — LOAD AND VALIDATE clean_master.csv
# =========================================================================
import os
import re
from typing import Optional, Sequence

import numpy as np
import pandas as pd
from scipy import optimize, stats

# The input contract. Exactly these columns, exactly these names. The
# validator does NOT coerce, rename, or guess: a schema violation raises.
EXPECTED_COLUMNS = [
    "sku_id", "month", "actual_units",
    "forecast_l1_units", "forecast_l2_units", "forecast_l3_units",
    "production_units", "stock_close_units", "stock_open_units",
    "sched_adherence", "yield_rate", "promo_flag",
]
NUMERIC_COLUMNS = [
    "actual_units", "forecast_l1_units", "forecast_l2_units",
    "forecast_l3_units", "production_units", "stock_close_units",
    "stock_open_units", "sched_adherence", "yield_rate",
]


def load_and_validate(path: str) -> pd.DataFrame:
    """
    Load clean_master.csv and enforce the input contract. Raises ValueError
    with a readable message on any violation. Nothing is coerced or repaired:
    a malformed input should be fixed upstream, not patched here.
    """
    df = pd.read_csv(path)
    problems = []

    # ---- columns: nothing missing, nothing extra -------------------------
    got, want = list(df.columns), EXPECTED_COLUMNS
    missing = [c for c in want if c not in got]
    unexpected = [c for c in got if c not in want]
    if missing:
        problems.append(f"MISSING column(s): {missing}")
    if unexpected:
        # Extra columns are refused rather than ignored. This notebook is a
        # blind estimator: silently tolerating an unknown column risks it
        # carrying information the estimator is not supposed to see.
        problems.append(
            f"UNEXPECTED column(s): {unexpected}. The input contract is exactly "
            f"{len(want)} columns; drop the extras before running."
        )
    if not missing and not unexpected and got != want:
        problems.append(f"Column ORDER differs from the contract.\n     expected: {want}\n     got:      {got}")

    if problems:  # stop here — dtype checks on a wrong schema are noise
        raise ValueError("clean_master.csv failed schema validation:\n  - "
                         + "\n  - ".join(problems))

    # ---- dtypes ----------------------------------------------------------
    for c in NUMERIC_COLUMNS:
        if not pd.api.types.is_numeric_dtype(df[c]):
            bad = df[c][pd.to_numeric(df[c], errors="coerce").isna() & df[c].notna()]
            problems.append(
                f"'{c}' is not numeric (dtype {df[c].dtype}). "
                f"{len(bad)} non-numeric value(s), e.g. {list(bad.unique()[:3])}"
            )

    if not pd.api.types.is_bool_dtype(df["promo_flag"]):
        vals = set(pd.Series(df["promo_flag"]).dropna().unique())
        if not vals <= {True, False, 0, 1, "True", "False", "true", "false"}:
            problems.append(
                f"'promo_flag' is not boolean and its values are not "
                f"boolean-like: {sorted(map(str, vals))[:6]}"
            )

    parsed_month = pd.to_datetime(df["month"], errors="coerce")
    if parsed_month.isna().any():
        n_bad = int(parsed_month.isna().sum())
        problems.append(f"'month' has {n_bad} unparseable value(s), "
                        f"e.g. {list(df.loc[parsed_month.isna(), 'month'].unique()[:3])}")

    if df["sku_id"].isna().any():
        problems.append(f"'sku_id' has {int(df['sku_id'].isna().sum())} null value(s)")

    if problems:
        raise ValueError("clean_master.csv failed dtype validation:\n  - "
                         + "\n  - ".join(problems))

    # ---- grain -----------------------------------------------------------
    df["month"] = parsed_month
    df["promo_flag"] = df["promo_flag"].astype(bool)
    dupes = df.duplicated(subset=["sku_id", "month"]).sum()
    if dupes:
        raise ValueError(
            f"Grain violation: {dupes} duplicate (sku_id, month) row(s). "
            "The table must have exactly one row per SKU per month."
        )

    return df.sort_values(["sku_id", "month"]).reset_index(drop=True)


# ---- locate the file -----------------------------------------------------
_candidates = [CSV_NAME, f"/content/{CSV_NAME}", f"/content/drive/MyDrive/{CSV_NAME}"]
_path = next((p for p in _candidates if os.path.exists(p)), None)
if _path is None:
    try:
        from google.colab import files  # type: ignore
        print(f"{CSV_NAME} not found in the session — please upload it now.")
        _path = next(iter(files.upload()))
    except ImportError:
        raise FileNotFoundError(
            f"{CSV_NAME} not found. Upload it to the working directory and re-run."
        )

clean_master = load_and_validate(_path)

print(f"VALIDATION PASSED — {os.path.basename(_path)}")
print(f"  rows   : {len(clean_master):,}")
print(f"  SKUs   : {clean_master['sku_id'].nunique()}")
print(f"  months : {clean_master['month'].nunique()} "
      f"({clean_master['month'].min():%Y-%m} to {clean_master['month'].max():%Y-%m})")
_nulls = clean_master.isna().sum()
print(f"  nulls  : {dict(_nulls[_nulls > 0]) if (_nulls > 0).any() else 'none'}")
clean_master.head()

---
## Methodology — reviewable without running anything

### 1. Why bias is estimated in **log space**

The obvious estimator, `mean((F − A) / A)`, is **not** an unbiased estimator of forecast bias when
demand is noisy. Write demand as `D = D̄·η` with multiplicative noise `E[η] = 1`. Even for a
forecast `F` exactly equal to `D̄` — no bias whatsoever — Jensen's inequality gives

$$\mathbb{E}\!\left[\frac{F}{D}\right] = \frac{F}{\bar D}\,\mathbb{E}\!\left[\frac{1}{\eta}\right] > \frac{F}{\bar D}\cdot\frac{1}{\mathbb{E}[\eta]} = 1$$

so the ratio estimator reports **apparent over-forecasting whose magnitude grows with demand
noise**. Percentage error is structurally asymmetric too: floored at −100%, unbounded above.

Bias is therefore estimated as `E[log(F / D)]`, which is zero under median-unbiased multiplicative
noise regardless of noise size. Reported in log units (`+0.10` ≈ +10.5% chronic over-forecast),
with a percentage column alongside.

### 2. Censoring — the Tobit approach

`actual_units` are **shipments, not demand**. When `stock_close_units` reaches zero, true demand
may have exceeded what could be shipped: that row is a **lower bound** on demand, not a
measurement. Both naive treatments fail in the *same* direction —

- **dropping** stockout rows selects away the highest-demand months;
- **trusting** `actual_units` truncates the right tail of demand.

So we fit **censored-normal maximum likelihood** (Tobit-type, with *observation-specific* limits:
each censored month carries its own threshold). Uncensored months contribute the normal
log-density; censored months contribute `log P(latent value beyond the observed bound)`.

### 3. The two models

**Model 1 — demand volatility (actuals only, no forecasts):**

$$\log D_t = \beta_0 + \beta_1\,\text{trend}_t + \beta_2\,\text{promo}_t + \varepsilon_t,\quad \varepsilon \sim N(0, \sigma_D^2)$$

Stockout month contributes `log P(log D_t ≥ log A_t)` — a **lower** bound.
Reported as `irreducible_volatility_cv = sqrt(exp(σ_D²) − 1)`.

**Model 2 — chronic bias and random error, per lag L:**

$$r_t = \log F_{L,t} - \log A_t = \text{bias}_L + u_t,\quad u \sim N(0, \sigma_L^2)$$

In a stockout month true `D` is *larger* than `A`, so true `r` is *smaller* than observed — an
**upper** bound, entering as `log P(r ≤ r_obs)`.

### 4. No circularity

`forecast_l*_units` appear **nowhere** in Model 1. If the forecast leaked into the demand estimate,
the bias estimate would partly compare the forecast with itself and shrink toward zero by
construction.

### 5. Reading `random_error_cv` honestly

It **necessarily contains demand noise** — a forecast made 1–3 months ahead cannot know that
month's shock. The three quantities are not orthogonal. The informative read is
`random_error_cv_lL` **versus** `irreducible_volatility_cv`: materially below → the forecast tracks
real demand movement; at or above → it is adding noise rather than information. Note that the ML
`σ` is downward-biased by roughly `sqrt((n−k)/n)`, so the volatility side of that comparison is
slightly optimistic; Cell 6 prints the correction.

### 6. Why the censoring correction is then *audited* rather than trusted

The correction treats stockout months as upper bounds on `log(F/A)`, which **mechanically** pulls
bias downward in proportion to stockout count. So a negative bias on a heavily stocked-out SKU has
two readings that look identical in the results table: genuine chronic under-forecasting (which
would *cause* the stockouts), or an artefact of the estimator. Cells 4–6 test this with two
independent lines of evidence and refuse to call it where they disagree.

### 7. Limitations, stated rather than papered over

- `stock_close_units == 0` is a **proxy** for a stockout; a month may end at zero having met demand
  exactly. Cell 5 measures how often that proxy actually binds.
- Demand can also be lost *before* stock hits exactly zero. `NEAR_ZERO_STOCK_QUANTILE` exposes a
  softer rule, off by default.
- **No monthly seasonal dummies** — with ~36 months per SKU, 11 extra parameters would fit noise.
  Stable seasonality therefore sits *inside* `irreducible_volatility_cv`, making it an **upper
  bound** on truly irreducible noise.
- Bias is one constant per lag, not split by promo vs non-promo.
- Errors are treated as independent across months; autocorrelation would make standard errors
  optimistic while leaving point estimates intact.

In [ ]:
# =========================================================================
# CELL 2 — THE ESTIMATOR
# =========================================================================

def _numerical_se(neg_loglik, theta: np.ndarray) -> np.ndarray:
    """
    Standard errors from the numerical Hessian of the negative log-likelihood
    at the optimum (the observed information matrix). Central finite
    differences; the parameter vector is at most 4 long, so cost is trivial.

    Returns NaN for any parameter where the Hessian is not invertible or the
    implied variance is negative — a flat or badly-conditioned likelihood
    should surface as "unknown", not as a confident small number.
    """
    theta = np.asarray(theta, dtype=float)
    n = theta.size
    eps = 1e-4 * np.maximum(np.abs(theta), 1.0)
    H = np.empty((n, n))
    for i in range(n):
        for j in range(n):
            tpp, tpm, tmp_, tmm = (theta.copy() for _ in range(4))
            tpp[i] += eps[i]; tpp[j] += eps[j]
            tpm[i] += eps[i]; tpm[j] -= eps[j]
            tmp_[i] -= eps[i]; tmp_[j] += eps[j]
            tmm[i] -= eps[i]; tmm[j] -= eps[j]
            H[i, j] = ((neg_loglik(tpp) - neg_loglik(tpm)
                        - neg_loglik(tmp_) + neg_loglik(tmm))
                       / (4.0 * eps[i] * eps[j]))
    H = 0.5 * (H + H.T)  # symmetrise away finite-difference asymmetry
    try:
        cov = np.linalg.inv(H)
        var = np.diag(cov)
        return np.where(var > 0, np.sqrt(np.abs(var)), np.nan)
    except np.linalg.LinAlgError:
        return np.full(n, np.nan)


def _censored_normal_mle(y, X, censored, bound: str, want_se: bool = False) -> dict:
    """
    Fit  y* = X @ beta + eps,  eps ~ N(0, sigma^2),  where y* is latent.

    censored == False : y* observed exactly.
    censored == True  : only a bound on y* is observed —
        bound == "lower":  y* >= y   (observed y understates the truth)
        bound == "upper":  y* <= y   (observed y overstates the truth)

    Tobit with an OBSERVATION-SPECIFIC limit: each censored row carries its
    own threshold (the value observed that month), not a shared global one.
    """
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float)
    censored = np.asarray(censored, dtype=bool)
    n, k = X.shape
    obs = ~censored

    # --- starting values: OLS on the uncensored rows only ---
    if obs.sum() >= k:
        beta0, *_ = np.linalg.lstsq(X[obs], y[obs], rcond=None)
        resid = y[obs] - X[obs] @ beta0
        sigma0 = float(np.std(resid, ddof=min(k, max(obs.sum() - 1, 1))))
    else:
        beta0 = np.zeros(k)
        beta0[0] = float(np.mean(y))
        sigma0 = float(np.std(y))
    if not np.isfinite(sigma0) or sigma0 <= 0:
        sigma0 = max(float(np.std(y)), 1e-3)

    # sigma optimised as log-sigma so it stays strictly positive
    def neg_loglik(theta):
        beta, log_sigma = theta[:k], theta[k]
        sigma = np.exp(log_sigma)
        z = (y - X @ beta) / sigma
        ll = np.empty(n)
        ll[obs] = stats.norm.logpdf(z[obs]) - log_sigma        # exact
        if censored.any():
            if bound == "lower":
                ll[censored] = stats.norm.logsf(z[censored])   # P(y* >= y)
            elif bound == "upper":
                ll[censored] = stats.norm.logcdf(z[censored])  # P(y* <= y)
            else:
                raise ValueError("bound must be 'lower' or 'upper'")
        total = float(np.sum(ll))
        return 1e12 if not np.isfinite(total) else -total

    theta0 = np.concatenate([beta0, [np.log(sigma0)]])
    bounds = [(None, None)] * k + [(np.log(1e-8), np.log(1e4))]
    res = optimize.minimize(neg_loglik, theta0, method="L-BFGS-B", bounds=bounds,
                            options={"maxiter": 2000, "ftol": 1e-10})

    sigma = float(np.exp(res.x[k]))
    at_bound = sigma <= 1e-7 or sigma >= 1e3   # pinned => flat/uninformative
    out = {"beta": res.x[:k], "sigma": sigma,
           "converged": bool(res.success) and not at_bound,
           "se_beta": np.full(k, np.nan)}
    if want_se and out["converged"]:
        out["se_beta"] = _numerical_se(neg_loglik, res.x)[:k]
    return out


def _lognormal_cv(sigma_log: float) -> float:
    """CV of a log-normal whose log has standard deviation sigma."""
    if not np.isfinite(sigma_log):
        return np.nan
    return np.inf if sigma_log > 20 else float(np.sqrt(np.expm1(sigma_log ** 2)))


def _infer_group(sku_ids) -> pd.Series:
    """
    Infer a grouping from the STRUCTURE of sku_id (e.g. 'A-01' -> 'A'): the
    leading non-digit run. No category names are supplied from outside the
    data. Returns all-NaN if the split is not useful (one group, or as many
    groups as SKUs), so a meaningless column is never silently created.
    """
    ids = pd.Series(list(sku_ids), dtype="object")
    prefixes = ids.map(
        lambda s: (m.group(1) if (m := re.match(r"^([^\W\d_]+)", str(s))) else np.nan)
    )
    ng = prefixes.nunique(dropna=True)
    if ng <= 1 or ng >= ids.nunique():
        return pd.Series(np.nan, index=ids.index, dtype="object")
    return prefixes


def estimate_demand_characteristics(
    df: pd.DataFrame,
    lags: Sequence = LAGS,
    min_uncensored: int = MIN_UNCENSORED,
    near_zero_stock_quantile: Optional[float] = NEAR_ZERO_STOCK_QUANTILE,
) -> pd.DataFrame:
    """
    Estimate demand volatility, chronic forecast bias and random forecast
    error per SKU from a (sku_id, month) panel.

    Returns one row per SKU. Estimates that cannot be trusted are returned as
    NaN with a reason in `notes` — no fallback value is ever substituted.
    """
    required = {"sku_id", "month", "actual_units", "stock_close_units"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"missing required columns: {sorted(missing)}")

    data = df.copy()
    data["month"] = pd.to_datetime(data["month"])
    data = data.sort_values(["sku_id", "month"])
    has_promo = "promo_flag" in data.columns
    rows = []

    for sku, g in data.groupby("sku_id", sort=True):
        g = g.reset_index(drop=True)
        note = []
        rec = {"sku_id": sku, "n_months": int(len(g))}

        actual = pd.to_numeric(g["actual_units"], errors="coerce").to_numpy(float)
        stock_close = pd.to_numeric(g["stock_close_units"], errors="coerce").to_numpy(float)

        # ---- censoring indicator ----
        # Hard rule: closing stock at or below zero => the month ended with
        # nothing left, so shipments are only a LOWER BOUND on demand.
        censored = np.nan_to_num(stock_close, nan=np.inf) <= 0
        if near_zero_stock_quantile is not None:
            pos = stock_close[np.isfinite(stock_close) & (stock_close > 0)]
            if pos.size:
                cut = np.quantile(pos, near_zero_stock_quantile)
                censored = censored | (np.isfinite(stock_close) & (stock_close <= cut))
                note.append(f"soft censoring rule active (q={near_zero_stock_quantile})")

        # Log space needs strictly positive shipments; a zero-shipment month
        # only says log D >= -inf, i.e. nothing — dropped rather than floored.
        usable = np.isfinite(actual) & (actual > 0)
        if (nd := int((~usable).sum())):
            note.append(f"{nd} month(s) dropped: actual_units missing or <= 0")

        rec["n_censored_months"] = int(censored[usable].sum())
        rec["n_usable_months"] = int(usable.sum())
        rec["censored_share"] = float(censored[usable].mean()) if usable.any() else np.nan
        n_unc = int((~censored[usable]).sum())

        # ---- Model 1: demand volatility (actuals only, NO forecasts) ----
        rec["irreducible_volatility_cv"] = np.nan
        rec["sigma_log_demand"] = np.nan
        rec["vol_df_correction"] = np.nan

        if n_unc < min_uncensored:
            note.append(f"volatility not estimated: only {n_unc} uncensored month(s), "
                        f"below min_uncensored={min_uncensored}")
        else:
            y = np.log(actual[usable])
            cols = [np.ones(y.size)]
            t = np.arange(y.size, dtype=float)
            t = (t - t.mean()) / (t.std() if t.std() > 0 else 1.0)
            cols.append(t)
            if has_promo:
                promo = (pd.to_numeric(g.loc[usable, "promo_flag"], errors="coerce")
                         .fillna(0).to_numpy(float))
                if 0 < promo.sum() < promo.size:      # include only if it varies
                    cols.append(promo)
            X = np.column_stack(cols)

            if y.size < X.shape[1] + 2:
                note.append("volatility not estimated: too few months for the model")
            else:
                fit = _censored_normal_mle(y, X, censored[usable], bound="lower")
                if fit["converged"]:
                    rec["sigma_log_demand"] = fit["sigma"]
                    rec["irreducible_volatility_cv"] = _lognormal_cv(fit["sigma"])
                    # ML sigma is downward-biased by ~sqrt((n-k)/n); the
                    # bias-corrected CV is reported separately rather than
                    # silently replacing the headline number.
                    k_par = X.shape[1]
                    if y.size > k_par:
                        adj = fit["sigma"] * np.sqrt(y.size / (y.size - k_par))
                        rec["vol_df_correction"] = _lognormal_cv(adj)
                else:
                    note.append("volatility not estimated: MLE did not converge")

        # ---- Model 2: chronic bias and random error, per lag ----
        for lag in lags:
            col = f"forecast_l{lag}_units"
            for suffix in ("", "_pct", "_se"):
                rec[f"chronic_bias{suffix}_l{lag}"] = np.nan
            rec[f"random_error_cv_l{lag}"] = np.nan

            if col not in g.columns:
                note.append(f"lag {lag}: column {col} absent")
                continue

            fc = pd.to_numeric(g[col], errors="coerce").to_numpy(float)
            ok = usable & np.isfinite(fc) & (fc > 0)
            if int(ok.sum()) == 0:
                note.append(f"lag {lag}: not estimated, no usable values in {col}")
                continue
            if int((~censored[ok]).sum()) < min_uncensored:
                note.append(f"lag {lag}: not estimated, "
                            f"{int((~censored[ok]).sum())} uncensored month(s)")
                continue

            # r = log(F / A). Where censored, true demand is LARGER than A, so
            # true r is SMALLER than observed: an upper bound, P(r <= obs).
            r = np.log(fc[ok]) - np.log(actual[ok])
            fit = _censored_normal_mle(r, np.ones((r.size, 1)), censored[ok],
                                       bound="upper", want_se=True)
            if fit["converged"]:
                b = float(fit["beta"][0])
                rec[f"chronic_bias_l{lag}"] = b                  # log units, signed
                rec[f"chronic_bias_pct_l{lag}"] = float(np.expm1(b))
                rec[f"chronic_bias_se_l{lag}"] = float(fit["se_beta"][0])
                rec[f"random_error_cv_l{lag}"] = _lognormal_cv(fit["sigma"])
            else:
                note.append(f"lag {lag}: MLE did not converge")

        if rec["censored_share"] == 1.0:
            note.append("every usable month is censored: results unreliable")

        rec["notes"] = "; ".join(note)
        rows.append(rec)

    out = pd.DataFrame(rows).set_index("sku_id")
    out.insert(0, "group", _infer_group(out.index).to_numpy())
    return out


print("Estimator defined.")

In [ ]:
# =========================================================================
# CELL 3 — RUN OVER THE FULL TABLE
# =========================================================================
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 300)

results = estimate_demand_characteristics(clean_master)

# The nine columns of the fixed downstream contract. This list must not grow:
# new diagnostics go to OUT_DIAGNOSTICS instead.
CONTRACT_COLS = ["irreducible_volatility_cv",
                 "chronic_bias_l1", "chronic_bias_l2", "chronic_bias_l3",
                 "random_error_cv_l1", "random_error_cv_l2", "random_error_cv_l3",
                 "n_censored_months", "notes"]
CORE_COLS = [c for c in CONTRACT_COLS if c != "notes"]

_tot_c = int(results["n_censored_months"].sum())
_tot_u = int(results["n_usable_months"].sum())

print(f"Estimated {len(results)} SKUs.")
print(f"Censoring incidence : {_tot_c:,} of {_tot_u:,} usable SKU-months "
      f"({_tot_c / max(_tot_u, 1):.1%})")
print(f"SKUs with any stockout month : {int((results['n_censored_months'] > 0).sum())}")
print(f"SKUs with a NaN estimate     : {int(results[CORE_COLS].isna().any(axis=1).sum())}")
print(f"SKUs carrying a note         : {int((results['notes'].str.len() > 0).sum())}")

if _tot_c == 0:
    print("\nNOTE: no censored months in this dataset. The censoring correction is a "
          "no-op here and Cells 4-6 will report nothing to adjudicate.")

results[CORE_COLS].round(4)

---
## Auditing the censoring correction

The correction treats a stockout month as an upper bound on `log(F/A)`, which **mechanically**
pulls estimated bias downward in proportion to stockout count. A negative bias on a heavily
stocked-out SKU therefore has two readings that are indistinguishable in the results table:

- **Real** — chronic under-forecasting causes under-production causes stockouts. Bias and stockouts
  share a cause and the correction is revealing it.
- **Artefact** — the correction produced the sign, and there is no underlying forecasting problem.

Two independent lines of evidence follow, then an adjudication that **refuses to call it** where
they disagree.

**Cell 4 — sensitivity.** Re-runs the identical estimator with closing stock forced positive, which
makes the correction a no-op. Without it, stockout months have depressed actuals, so `log(F/A)` is
*inflated* and raw bias should skew **positive**. If raw bias is already meaningfully negative, the
finding survives the naive method and is not a creature of the correction.

*Magnitude, not sign.* An earlier version of this notebook split on the bare sign of raw bias and
labelled values like −0.004 as confirmation. That is noise. The test here requires raw bias to sit
at least `BIAS_SE_MULTIPLIER` standard errors from zero, where the standard error comes from the
observed information matrix of that SKU's own fit — derived per SKU, not chosen by eye.

**Cell 5 — does the stockout flag actually bind?** The entire question reduces to whether
`stock_close_units == 0` really meant lost demand. In a genuinely binding stockout, shipments should
consume essentially everything available (`stock_open + production`). The shipment ratio in censored
months is compared against **the same SKU's own** ratios in its non-censored months, so no absolute
cut-off is imposed.

**Cell 6 — adjudication.** `REAL`, `ARTEFACT_RISK`, or `INCONCLUSIVE`. The third bucket is not a
failure of the method; it is the honest output when two valid tests point different ways, and it
exists so that a weak finding cannot be promoted to an action item by default.

In [ ]:
# =========================================================================
# CELL 4 — DIAGNOSTIC 1: CENSORING SENSITIVITY
# =========================================================================
# Re-run the SAME estimator with every month marked fully stocked, disabling
# the censoring correction entirely. Any difference between the runs is
# attributable to the correction and nothing else.
_uncensored_run = estimate_demand_characteristics(
    clean_master.assign(stock_close_units=1.0)
)

sensitivity = pd.DataFrame({
    "n_cens":         results["n_censored_months"],
    "cens_share":     results["censored_share"],
    "bias_corrected": results["chronic_bias_l1"],
    "bias_raw":       _uncensored_run["chronic_bias_l1"],
    "bias_raw_se":    _uncensored_run["chronic_bias_se_l1"],
    "vol_corrected":  results["irreducible_volatility_cv"],
    "vol_raw":        _uncensored_run["irreducible_volatility_cv"],
})
sensitivity["shift"] = sensitivity["bias_corrected"] - sensitivity["bias_raw"]

# ---- materiality threshold, derived from THIS dataset --------------------
# Median censored-month count among SKUs that have any stockouts at all. On a
# lightly-censored dataset this is small; on a heavily-censored one it rises.
_with_cens = sensitivity.loc[sensitivity["n_cens"] > 0, "n_cens"]
if CENS_MATERIALITY_OVERRIDE is not None:
    CENS_MATERIAL = float(CENS_MATERIALITY_OVERRIDE)
    _mat_src = "CONFIG override"
elif len(_with_cens):
    CENS_MATERIAL = float(_with_cens.median())
    _mat_src = "median censored-month count among SKUs with any stockouts"
else:
    CENS_MATERIAL = np.inf
    _mat_src = "no censored months in this dataset"

# ---- is raw bias meaningfully signed, or indistinguishable from zero? ----
# Uses each SKU's own standard error from the observed information matrix.
# Where the SE is unavailable (flat likelihood), the verdict is "unknown"
# rather than a default to either side.
def _raw_sign(r):
    b, se = r["bias_raw"], r["bias_raw_se"]
    if not np.isfinite(b):
        return "unknown"
    if not np.isfinite(se) or se <= 0:
        return "unknown"
    k = BIAS_SE_MULTIPLIER
    if b + k * se < 0:
        return "negative"
    if b - k * se > 0:
        return "positive"
    return "indistinguishable"

sensitivity["raw_sign"] = sensitivity.apply(_raw_sign, axis=1)

# ---- shift magnitudes: censored SKUs vs everyone -------------------------
# Reporting a single all-SKU median conflates two different things: on a
# lightly-censored dataset the zero-stockout majority drags it to 0.0000 and
# hides how large the correction is where it actually applies.
_cens_skus = sensitivity["n_cens"] > 0
SHIFT_CENSORED = sensitivity.loc[_cens_skus, "shift"].abs().median()
SHIFT_ALL = sensitivity["shift"].abs().median()
SHIFT_ZERO = sensitivity.loc[~_cens_skus, "shift"].abs().median()

print(f"Materiality threshold : >= {CENS_MATERIAL:.0f} censored months ({_mat_src})")
print(f"SKUs adjudicated      : {int((sensitivity['n_cens'] >= CENS_MATERIAL).sum())}")
print()
print("MEDIAN ABSOLUTE SHIFT FROM THE CORRECTION (log units)")
_fmt = lambda v: "n/a" if not np.isfinite(v) else f"{v:.4f}"
print(f"  among SKUs WITH stockouts  : {_fmt(SHIFT_CENSORED)}   <- the meaningful number")
print(f"  across all SKUs            : {_fmt(SHIFT_ALL)}   (diluted by zero-stockout SKUs)")
print(f"  among zero-stockout SKUs   : {_fmt(SHIFT_ZERO)}   (must be ~0 — sanity check)")
if np.isfinite(SHIFT_ZERO) and SHIFT_ZERO > 1e-6:
    print("  WARNING: non-zero shift where no month was censored. The diagnostic "
          "itself is suspect; do not rely on this section.")
print()
print("RAW BIAS SIGN (uncorrected, vs its own standard error, "
      f"k={BIAS_SE_MULTIPLIER})")
print(sensitivity.loc[_cens_skus, "raw_sign"].value_counts().to_string()
      if _cens_skus.any() else "  no censored SKUs")
print()
print("Top 20 SKUs by censored-month count:")
print(sensitivity.sort_values("n_cens", ascending=False).head(20)[
    ["n_cens", "bias_corrected", "bias_raw", "bias_raw_se", "raw_sign", "shift"]
].round(4).to_string())

In [ ]:
# =========================================================================
# CELL 5 — DIAGNOSTIC 2: DID THE STOCKOUT FLAG ACTUALLY BIND?
# =========================================================================
# The censoring correction is only justified where stock_close_units == 0
# genuinely meant demand could not be met. In a binding stockout, shipments
# should consume essentially everything available that month.
#
#     available    = stock_open_units + production_units
#     ship_ratio   = actual_units / available
#
# A ratio at the ceiling means the shelf emptied. A ratio well below it means
# closing stock reached zero WITHOUT demand being constrained -- in which case
# the flag is over-firing and the correction is being applied where it should
# not be.
#
# The ceiling is defined per SKU, from that SKU's own non-censored months, so
# no absolute cut-off is imposed on datasets with different stocking policies.

_m = clean_master.copy()
_m["available"] = _m["stock_open_units"] + _m["production_units"]
_m["is_censored"] = _m["stock_close_units"] <= 0
_m["ship_ratio"] = np.where(
    (_m["available"] > 0) & np.isfinite(_m["available"]),
    _m["actual_units"] / _m["available"], np.nan)

_rows = []
for _sku, _g in _m.groupby("sku_id", sort=True):
    _c = _g.loc[_g["is_censored"] & _g["ship_ratio"].notna(), "ship_ratio"]
    _u = _g.loc[~_g["is_censored"] & _g["ship_ratio"].notna(), "ship_ratio"]
    # Reference ceiling: the SKU's own high-water mark in months when it did
    # NOT stock out. Requires enough uncensored months to locate a quantile.
    _ref = float(_u.quantile(BINDING_RATIO_QUANTILE)) if len(_u) >= MIN_UNCENSORED else np.nan
    _rows.append({
        "sku_id": _sku,
        "n_cens_measurable": int(len(_c)),
        "ship_ratio_cens_median": float(_c.median()) if len(_c) else np.nan,
        "ship_ratio_cens_min": float(_c.min()) if len(_c) else np.nan,
        "ship_ratio_ref": _ref,
        "share_cens_at_ceiling": (float((_c >= _ref).mean())
                                  if len(_c) and np.isfinite(_ref) else np.nan),
    })

binding = pd.DataFrame(_rows).set_index("sku_id")

def _binding_verdict(r):
    if r["n_cens_measurable"] == 0:
        return "no measurable censored months"
    if not np.isfinite(r["share_cens_at_ceiling"]):
        return "unknown (no reference distribution)"
    if r["share_cens_at_ceiling"] >= 0.5:
        return "BINDING (correction justified)"
    if r["share_cens_at_ceiling"] <= 0.2:
        return "NOT BINDING (flag over-firing)"
    return "MIXED"

binding["binding_verdict"] = binding.apply(_binding_verdict, axis=1)

# NOTE ON THE 0.5 / 0.2 SPLIT: these are majority / small-minority marks on a
# share, not tuned magic numbers -- "most censored months hit the ceiling" vs
# "hardly any did". They are stated here rather than hidden in CONFIG because
# changing them changes the meaning of the verdict, not just its sensitivity.

_meas = binding[binding["n_cens_measurable"] > 0]
print(f"SKUs with measurable censored months: {len(_meas)} of {len(binding)}")
if len(_meas):
    print(f"(a censored month is measurable only when stock_open_units and "
          f"production_units are both present — the first month of the panel "
          f"has no opening stock)")
    print()
    print("Reference ceiling = each SKU's own "
          f"{BINDING_RATIO_QUANTILE:.0%} quantile of ship_ratio in NON-censored months.")
    print()
    print(_meas["binding_verdict"].value_counts().to_string())
    print()
    print(_meas.sort_values("n_cens_measurable", ascending=False).head(20).round(4).to_string())
else:
    print("No censored months could be measured. Where this is because the dataset has no "
          "stockouts at all, there is nothing to audit and Cell 6 will say so. Where "
          "stockouts exist but stock_open_units/production_units are missing for them, "
          "Cell 6 loses one line of evidence and will mark affected SKUs INCONCLUSIVE.")

In [ ]:
# =========================================================================
# CELL 6 — ADJUDICATION: REAL / ARTEFACT_RISK / INCONCLUSIVE
# =========================================================================
# Combines the two independent diagnostics. The rule is deliberately
# conservative: a SKU reaches REAL only when BOTH lines of evidence agree, and
# anything else that is not clearly an artefact is left INCONCLUSIVE rather
# than promoted. A finding that cannot survive its own audit should not appear
# on an action list by default.

adjudication = sensitivity.join(binding, how="left")

def _adjudicate(r):
    # Only SKUs whose corrected bias is negative AND materially censored are
    # in scope: this audit exists to test negative signs produced under
    # censoring. Everything else needs no adjudication.
    if not np.isfinite(r["bias_corrected"]):
        return "n/a (no estimate)"
    if r["n_cens"] < CENS_MATERIAL:
        return "n/a (below materiality)"
    if r["bias_corrected"] >= 0:
        return "n/a (bias not negative)"

    sign = r["raw_sign"]
    bind = r.get("binding_verdict", "unknown")
    binding_ok = str(bind).startswith("BINDING")
    binding_bad = str(bind).startswith("NOT BINDING")

    # Both lines agree the finding is real: raw bias is meaningfully negative
    # WITHOUT the correction, and the stockouts genuinely bound.
    if sign == "negative" and binding_ok:
        return "REAL"
    # Both lines agree it is an artefact: raw bias gives no negative signal,
    # and the stockout flag is not even binding.
    if sign in ("positive", "indistinguishable") and binding_bad:
        return "ARTEFACT_RISK"
    # Everything else -- one test silent, tests disagreeing, SE unavailable.
    return "INCONCLUSIVE"

adjudication["verdict"] = adjudication.apply(_adjudicate, axis=1)

def _why(r):
    """Plain-language evidence string, so the verdict is auditable per SKU."""
    if str(r["verdict"]).startswith("n/a"):
        return ""
    return (f"raw bias {r['bias_raw']:+.4f} +/- {r['bias_raw_se']:.4f} "
            f"({r['raw_sign']}); stockouts {r.get('binding_verdict', 'unknown')}")

adjudication["evidence"] = adjudication.apply(_why, axis=1)

_scope = adjudication[~adjudication["verdict"].str.startswith("n/a")]
print(f"SKUs in scope for adjudication: {len(_scope)}")
print()
if len(_scope):
    print(adjudication["verdict"].value_counts().to_string())
    print()
    print(_scope.sort_values("n_cens", ascending=False)[
        ["n_cens", "bias_corrected", "bias_raw", "bias_raw_se",
         "share_cens_at_ceiling", "verdict"]].round(4).to_string())
else:
    print("Nothing to adjudicate: no SKU has both material censoring and a "
          "negative corrected bias. The censoring correction is not driving "
          "any finding in this dataset.")

In [ ]:
# =========================================================================
# CELL 7 — OUTPUT SUMMARY  (plain text, copy-paste friendly)
# =========================================================================
_L = 100
_lines = []
_add = _lines.append

_add("=" * _L)
_add("DEMAND CHARACTERISTICS — PER-SKU ESTIMATES")
_add("=" * _L)
_add(f"Source file : {os.path.basename(_path)}")
_add(f"Panel       : {len(clean_master):,} rows | {clean_master['sku_id'].nunique()} SKUs | "
     f"{clean_master['month'].nunique()} months "
     f"({clean_master['month'].min():%Y-%m} to {clean_master['month'].max():%Y-%m})")
_add(f"Config      : min_uncensored={MIN_UNCENSORED}, "
     f"near_zero_stock_quantile={NEAR_ZERO_STOCK_QUANTILE}, "
     f"bias_se_multiplier={BIAS_SE_MULTIPLIER}, "
     f"binding_ratio_quantile={BINDING_RATIO_QUANTILE}")
_add("")
_add("WHAT WAS ESTIMATED")
_add("  irreducible_volatility_cv : CV of unpredictable demand noise, after removing level,")
_add("                              linear trend and planned promo effects.")
_add("  chronic_bias_lN           : persistent signed forecast bias at lag N, in LOG units.")
_add("                              POSITIVE = chronic OVER-forecast. 0.10 ~ +10.5%.")
_add("  random_error_cv_lN        : dispersion of forecast error around its own chronic level.")
_add("  n_censored_months         : months ending at zero stock, where shipments are only a")
_add("                              LOWER BOUND on true demand.")
_add("")
_add("KEY METHODOLOGY CHOICES")
_add("  1. Bias estimated as E[log(F/A)], NOT mean((F-A)/A). The ratio form is biased under")
_add("     multiplicative demand noise (Jensen): it manufactures apparent over-forecast that")
_add("     grows with volatility, even when true bias is zero.")
_add("  2. Stockout months handled by CENSORED-NORMAL MLE (Tobit-type, observation-specific")
_add("     limits), not dropped and not taken at face value — both cut off the high-demand tail.")
_add("  3. Forecast columns are used NOWHERE in the volatility estimate, so the bias estimate is")
_add("     not a comparison of the forecast with itself.")
_add("  4. NO seasonal dummies (~36 months/SKU cannot support 11 extra parameters), so stable")
_add("     seasonality remains INSIDE irreducible_volatility_cv — treat it as an UPPER BOUND.")
_add("  5. random_error_cv necessarily CONTAINS demand noise: a forecast made 1-3 months out")
_add("     cannot know that month's shock. Compare random_error_cv_lN against")
_add("     irreducible_volatility_cv — materially below = the forecast carries information.")
_add("  6. Nothing hard-coded from outside the data. All tunables are in CELL 0.")
_add("")

_tot_c = int(results["n_censored_months"].sum())
_tot_u = int(results["n_usable_months"].sum())
_add(f"Censoring incidence : {_tot_c:,} of {_tot_u:,} usable SKU-months "
     f"({_tot_c / max(_tot_u, 1):.1%})")
_add(f"SKUs with a NaN estimate : {int(results[CORE_COLS].isna().any(axis=1).sum())}")
_add(f"SKUs carrying a note     : {int((results['notes'].str.len() > 0).sum())}")
_add("")

_add("-" * _L)
_add("FULL PER-SKU RESULTS   (bias in log units; positive = chronic OVER-forecast)")
_add("-" * _L)
_add(results[CORE_COLS].round(4).to_string())
_add("")

_add("-" * _L)
_add("SAME BIAS FIGURES AS PERCENTAGES  (exp(bias) - 1)")
_add("-" * _L)
_add((results[[f"chronic_bias_pct_l{l}" for l in LAGS]] * 100).round(1).to_string())
_add("")

_add("-" * _L)
_add("GROUP-LEVEL MEDIANS")
_add("-" * _L)
if results["group"].notna().any():
    _add("Grouping inferred from the STRUCTURE of sku_id (leading letters) only — no category")
    _add("names were supplied. Medians, not means, to limit leverage from single odd SKUs.")
    _add("")
    _grp = results.groupby("group")[CORE_COLS[:-1]].median()
    _grp.insert(0, "n_skus", results.groupby("group").size())
    _grp.insert(1, "n_cens_months", results.groupby("group")["n_censored_months"].sum())
    _add(_grp.round(4).to_string())
    _add("")
    _add("Watch the n_cens_months column: a large imbalance between groups is a finding in its")
    _add("own right, and one that does not depend on the estimator at all.")
else:
    _add("No natural grouping detected in sku_id — per-SKU results only.")
_add("")
_add("Portfolio median (all SKUs):")
_add(results[CORE_COLS[:-1]].median().round(4).to_string())
_add("")
_add("FORECAST VALUE CHECK — random_error_cv vs irreducible_volatility_cv")
_add("  A trend-plus-promo baseline fit on this same data has dispersion equal to the volatility")
_add("  figure. Where random_error_cv exceeds it, the live forecast is noisier than that")
_add("  baseline. CAVEAT: the baseline is fit IN-SAMPLE on the same months while the forecast is")
_add("  genuinely ex-ante, so this is indicative, not a clean horse race. A rolling-origin")
_add("  backtest of the baseline is what would settle it.")
_vol_med = results["irreducible_volatility_cv"].median()
_vol_adj = results["vol_df_correction"].median()
_add(f"    volatility median (ML)                : {_vol_med:.4f}")
_add(f"    volatility median (df-corrected)      : {_vol_adj:.4f}   <- the fairer baseline")
for _l in LAGS:
    _re = results[f"random_error_cv_l{_l}"].median()
    _verdict = "forecast NOISIER than baseline" if _re > _vol_adj else "forecast beats baseline"
    _add(f"    random_error_cv_l{_l} median           : {_re:.4f}   {_verdict}")
_add("")

_add("-" * _L)
_add("CENSORING AUDIT — IS THE CORRECTION FINDING BIAS OR CREATING IT?")
_add("-" * _L)
_add("The correction treats a stockout month as an upper bound on log(F/A), which MECHANICALLY")
_add("pulls bias downward in proportion to stockout count. Two independent tests follow.")
_add("")
_add(f"Materiality threshold : >= {CENS_MATERIAL:.0f} censored months ({_mat_src})")
_add("")
_add("TEST 1 — sensitivity. Re-run with the correction disabled. Without it, stockout months")
_add("  have depressed actuals, so log(F/A) is INFLATED and raw bias should skew POSITIVE.")
_add("  Raw bias is called 'negative' only when it sits at least "
     f"{BIAS_SE_MULTIPLIER} standard error(s) below")
_add("  zero — a magnitude test, not a sign test, using each SKU's own standard error.")
_fs = lambda v: "n/a" if not np.isfinite(v) else f"{v:.4f}"
_add(f"    median |shift| among SKUs WITH stockouts : {_fs(SHIFT_CENSORED)} log units")
_add(f"    median |shift| across all SKUs           : {_fs(SHIFT_ALL)} (diluted by zero-stockout SKUs)")
_add(f"    median |shift| among zero-stockout SKUs  : {_fs(SHIFT_ZERO)} (must be ~0 — sanity check)")
_add("")
_add("TEST 2 — did the stockout flag actually bind? In a genuine stockout, shipments should")
_add("  consume nearly all available stock (opening + production). Each SKU's censored months are")
_add(f"  compared against its OWN {BINDING_RATIO_QUANTILE:.0%} quantile of ship_ratio in non-censored months.")
_add("")

_scope = adjudication[~adjudication["verdict"].str.startswith("n/a")]
if len(_scope) == 0:
    _add("NOTHING TO ADJUDICATE. No SKU has both material censoring and a negative corrected")
    _add("bias, so the censoring correction is not driving any finding in this dataset.")
else:
    _add(f"ADJUDICATION — {len(_scope)} SKU(s) in scope "
         f"(materially censored AND negative corrected bias)")
    for _v in ["REAL", "INCONCLUSIVE", "ARTEFACT_RISK"]:
        _add(f"    {_v:<15} : {int((adjudication['verdict'] == _v).sum())}")
    _add("")
    _add(_scope.sort_values("n_cens", ascending=False)[
        ["n_cens", "bias_corrected", "bias_raw", "bias_raw_se",
         "share_cens_at_ceiling", "verdict"]].round(4).to_string())
    _add("")
    for _v, _hdr in [
        ("REAL", "SURVIVED THE AUDIT — negative bias holds without the correction, and the "
                 "stockouts genuinely bound.\nThis is a SCREENING result at "
                 f"k={BIAS_SE_MULTIPLIER} standard error(s), roughly a 1-in-6 false-positive rate "
                 "per SKU under\nthe null of zero bias. Treat as 'worth investigating', not "
                 "'proven':"),
        ("INCONCLUSIVE", "EVIDENCE DOES NOT SETTLE IT — the two tests disagree, or one is silent. "
                         "Do NOT action these\nwithout further investigation, and do NOT report "
                         "them as findings:"),
        ("ARTEFACT_RISK", "LIKELY ARTEFACT — the negative sign appears only after correction, and "
                          "the stockout flag is\nnot binding. Treat as an estimator effect, not a "
                          "forecasting problem:"),
    ]:
        _sub = adjudication[adjudication["verdict"] == _v]
        if len(_sub):
            _add(_hdr)
            for _s in _sub.index:
                _add(f"   {_s:<10} n_cens={int(_sub.loc[_s, 'n_cens']):>3}  "
                     f"{_sub.loc[_s, 'evidence']}")
            _add("")

_add("SKUs whose findings are UNAFFECTED by any of this: those with zero censored months. For")
_add("them the correction is a no-op and the estimates stand on their own.")
_add(f"    zero-censoring SKUs : {int((results['n_censored_months'] == 0).sum())} of {len(results)}")
_add("")

_add("-" * _L)
_add("SKUs FLAGGED, WITH REASON")
_add("-" * _L)
_flagged = results[results["notes"].str.len() > 0]
_n_nan = int(results[CORE_COLS].isna().any(axis=1).sum())
if len(_flagged) == 0:
    _add("None. Every SKU converged with no excluded months.")
else:
    if _n_nan:
        _add(f"{_n_nan} SKU(s) returned NaN for at least one estimate — NOT reported, "
             f"deliberately.")
        _add("No fallback value was substituted anywhere.")
    else:
        _add("No estimate returned NaN: every SKU converged. The notes below are data-quality")
        _add("observations (e.g. months excluded from the fit), not failed estimates.")
    _add("")
    for _sku, _row in _flagged.iterrows():
        _add(f"  {_sku:<12} (censored {_row['censored_share']:.0%} of "
             f"{int(_row['n_usable_months'])} usable months)")
        for _n in str(_row["notes"]).split("; "):
            _add(f"       - {_n}")
_add("")
_add("=" * _L)
_add("END OF SUMMARY")
_add("=" * _L)

SUMMARY_TEXT = "\n".join(_lines)
print(SUMMARY_TEXT)

In [ ]:
# =========================================================================
# CELL 8 — WRITE OUTPUTS
# =========================================================================
# Two files, deliberately separated:
#
#   OUT_PRIMARY      — the fixed downstream contract. Exactly nine columns.
#                      This list MUST NOT GROW as new diagnostics are added;
#                      that is the whole point of the second file.
#   OUT_DIAGNOSTICS  — everything else: diagnostic columns, verdicts,
#                      evidence. Free to evolve.

primary = results[CONTRACT_COLS].copy()
assert list(primary.columns) == CONTRACT_COLS, "primary output contract violated"
assert primary.index.name == "sku_id", "primary output must be indexed by sku_id"
primary.to_csv(OUT_PRIMARY)

diagnostics = (
    adjudication
    .join(results[["group", "n_months", "n_usable_months", "censored_share",
                   "sigma_log_demand", "vol_df_correction"]
                  + [f"chronic_bias_se_l{l}" for l in LAGS]
                  + [f"chronic_bias_pct_l{l}" for l in LAGS]],
          how="left", rsuffix="_res")
)
diagnostics.index.name = "sku_id"
diagnostics.to_csv(OUT_DIAGNOSTICS)

# The printed summary is also saved, so a reviewer can read the reasoning
# without re-running the notebook.
with open("demand_characteristics_summary.txt", "w") as _f:
    _f.write(SUMMARY_TEXT)

print(f"WROTE {OUT_PRIMARY}")
print(f"   {primary.shape[0]} rows x {primary.shape[1]} columns (fixed contract)")
print(f"   columns: {list(primary.columns)}")
print()
print(f"WROTE {OUT_DIAGNOSTICS}")
print(f"   {diagnostics.shape[0]} rows x {diagnostics.shape[1]} columns (may evolve)")
print()
print("WROTE demand_characteristics_summary.txt")
print()
print("Colab: download via the Files pane (left sidebar), or run")
print("   from google.colab import files; files.download(OUT_PRIMARY)")

---
## Appendix — method validation on **simulated** data

The cell below does **not** touch your data. It generates a synthetic panel with *known* bias and
volatility, then checks whether the estimator recovers numbers it was never told — and how far the
two methods this notebook deliberately avoids drift under the same censoring.

**The figures it prints describe the method, not your SKUs.** Do not copy them into a results
review. Set `RUN_METHOD_VALIDATION = False` in Cell 0 to skip.

In [ ]:
# =========================================================================
# APPENDIX — RECOVERY TEST ON SIMULATED DATA (not your results)
# =========================================================================
if not RUN_METHOD_VALIDATION:
    print("Skipped (RUN_METHOD_VALIDATION = False).")
else:
    _rng = np.random.default_rng(7)
    _TRUE_SIGMA = {"A": 0.30, "B": 0.55}          # true log-sd of demand
    _TRUE_BIAS = {1: 0.00, 2: 0.12, 3: -0.20}     # true chronic bias, log units

    _rows = []
    for _grp, _sigma in _TRUE_SIGMA.items():
        for _i in range(1, 16):
            _level = np.log(_rng.uniform(20_000, 120_000))
            _cap = np.exp(_level + _rng.uniform(-0.15, 0.35))   # tight enough to stock out
            _stock = _cap * 0.4
            for _t in range(36):
                _promo = bool(_rng.random() < 0.3)
                _mu = _level + 0.25 * _promo
                _demand = np.exp(_mu + _rng.normal(0, _sigma))
                _prod = _cap * _rng.uniform(0.85, 1.15)
                _avail = _stock + _prod
                _ship = min(_demand, _avail)
                _open = _stock
                _stock = _avail - _ship
                _row = {"sku_id": f"{_grp}-{_i:02d}",
                        "month": pd.Timestamp("2023-01-01") + pd.DateOffset(months=_t),
                        "actual_units": _ship, "production_units": _prod,
                        "stock_close_units": _stock, "stock_open_units": _open,
                        "sched_adherence": 0.92, "yield_rate": 0.985,
                        "promo_flag": _promo}
                for _lag, _b in _TRUE_BIAS.items():
                    # forecast targets the conditional MEDIAN, offset by the
                    # true bias, plus its own independent noise
                    _row[f"forecast_l{_lag}_units"] = np.exp(_mu + _b + _rng.normal(0, 0.18))
                _rows.append(_row)

    _sim = pd.DataFrame(_rows)
    _res = estimate_demand_characteristics(_sim)
    _res["grp"] = [s.split("-")[0] for s in _res.index]

    print("SIMULATED DATA — validates the METHOD, says nothing about your dataset.")
    print(f"Stockout months in simulation: {(_sim.stock_close_units <= 0).mean():.1%}\n")

    print("VOLATILITY RECOVERY   (true CV = sqrt(exp(sigma^2) - 1))")
    for _grp, _sigma in _TRUE_SIGMA.items():
        _true_cv = np.sqrt(np.expm1(_sigma ** 2))
        _est = _res.loc[_res.grp == _grp, "irreducible_volatility_cv"].median()
        print(f"   group {_grp}:  true {_true_cv:.3f}   estimated (median) {_est:.3f}")

    print("\nBIAS RECOVERY, log units")
    print(f"{'lag':>4} {'true':>8} {'censored MLE':>14} "
          f"{'naive (F-A)/A':>15} {'stockouts dropped':>19}")
    for _lag, _b in _TRUE_BIAS.items():
        _mle = _res[f"chronic_bias_l{_lag}"].median()
        _f = _sim[f"forecast_l{_lag}_units"].to_numpy()
        _a = _sim["actual_units"].to_numpy()
        _naive = np.log1p(np.mean((_f - _a) / _a))          # ratio trap, on log scale
        _keep = _sim["stock_close_units"].to_numpy() > 0
        _drop = np.mean(np.log(_f[_keep] / _a[_keep]))      # log space, selection bias
        print(f"{_lag:>4} {_b:>8.3f} {_mle:>14.3f} {_naive:>15.3f} {_drop:>19.3f}")

    print("\nThe two right-hand columns are what you would have reported using the methods")
    print("this notebook deliberately avoids. Both overstate over-forecasting.")

    # ---------------------------------------------------------------------
    # CALIBRATION: how does the correction behave as censoring intensifies?
    # ---------------------------------------------------------------------
    # The Tobit model assumes the censoring threshold is EXOGENOUS. Here it is
    # not: a month is censored precisely BECAUSE demand was high, and high
    # demand means low true r = log(F/D). The bound is correlated with the
    # latent value it bounds. That is endogenous censoring, and it makes the
    # correction OVERSHOOT downward — increasingly so as censoring intensifies.
    #
    # This is a real limitation of the method, not of this dataset, so it is
    # measured here rather than left as a caveat in prose.
    print("\n" + "-" * 78)
    print("CALIBRATION — estimated minus TRUE bias, by censoring intensity")
    print("-" * 78)

    _rng2 = np.random.default_rng(21)
    _TRUE_B = 0.05                      # one known bias for all SKUs
    _cal_rows = []
    for _slack_i, _slack in enumerate([1.20, 0.55, 0.30, 0.10, -0.15, -0.45]):
        for _i in range(12):
            _lvl = np.log(_rng2.uniform(20_000, 120_000))
            _cap = np.exp(_lvl + _slack)      # tighter slack => more stockouts
            _stock = _cap * 0.5
            for _t in range(36):
                _promo = bool(_rng2.random() < 0.3)
                _mu = _lvl + 0.2 * _promo
                _d = np.exp(_mu + _rng2.normal(0, 0.30))
                _prod = _cap * _rng2.uniform(0.85, 1.15)
                _avail = _stock + _prod
                _ship = min(_d, _avail)
                _open = _stock
                _stock = _avail - _ship
                _cal_rows.append({
                    "sku_id": f"S{_slack_i}-{_i:02d}",
                    "month": pd.Timestamp("2023-01-01") + pd.DateOffset(months=_t),
                    "actual_units": _ship, "production_units": _prod,
                    "stock_close_units": _stock, "stock_open_units": _open,
                    "sched_adherence": 0.92, "yield_rate": 0.985, "promo_flag": _promo,
                    "forecast_l1_units": np.exp(_mu + _TRUE_B + _rng2.normal(0, 0.18)),
                    "forecast_l2_units": np.exp(_mu + _TRUE_B + _rng2.normal(0, 0.18)),
                    "forecast_l3_units": np.exp(_mu + _TRUE_B + _rng2.normal(0, 0.18)),
                })
    _cal = pd.DataFrame(_cal_rows)
    _cal_res = estimate_demand_characteristics(_cal)
    _cal_raw = estimate_demand_characteristics(_cal.assign(stock_close_units=1.0))
    _cal_res["cens_rate"] = _cal_res["censored_share"]
    _cal_res["bias_raw"] = _cal_raw["chronic_bias_l1"]
    _bins = pd.cut(_cal_res["cens_rate"], [-0.001, 0.02, 0.10, 0.25, 0.50, 1.01],
                   labels=["0-2%", "2-10%", "10-25%", "25-50%", ">50%"])

    print(f"True bias in every simulated SKU: {_TRUE_B:+.3f} log units\n")
    _tab = pd.DataFrame({
        "n_skus": _cal_res.groupby(_bins, observed=True).size(),
        "corrected_est": _cal_res.groupby(_bins, observed=True)["chronic_bias_l1"].median(),
        "raw_est": _cal_res.groupby(_bins, observed=True)["bias_raw"].median(),
    })
    _tab["corrected_error"] = _tab["corrected_est"] - _TRUE_B
    _tab["raw_error"] = _tab["raw_est"] - _TRUE_B
    print(_tab.round(4).to_string())
    print()
    print("HOW TO READ THIS TABLE.")
    print("  * raw_error grows sharply with censoring — this is the selection effect the")
    print("    correction exists to remove, and at heavy censoring it is severe.")
    print("  * corrected_error stays within a modest band at every intensity, but does NOT")
    print("    converge to zero and does not move in a consistent direction. Treat it as a")
    print("    band of residual uncertainty on the corrected estimate, not as an offset to")
    print("    subtract.")
    print("  * PRACTICAL RULE: on a heavily-censored SKU, a corrected bias smaller in")
    print("    magnitude than the corrected_error in its band should not be trusted for its")
    print("    SIGN. That is precisely the situation Cell 6 marks INCONCLUSIVE.")
    print()
    print("  Why residual error remains: the Tobit model assumes an EXOGENOUS censoring")
    print("  threshold, but here a month is censored precisely BECAUSE demand was high —")
    print("  which is exactly when true log(F/D) is low. The bound correlates with the latent")
    print("  value it bounds. The correction removes most of the resulting distortion; it")
    print("  cannot remove all of it.")
    print()
    print("  CAVEAT: bands with few SKUs are themselves noisy — read n_skus before leaning on")
    print("  any single row, and note this is ONE simulated draw, not a full Monte Carlo.")